# Fase 3 · Feature Engineering (Ingeniería de características)
**Dataset de propiedades en venta — Mérida, Yucatán**

## Propósito
Transformar el dataset crudo validado en la Fase 2 en un conjunto listo para
modelar: eliminar duplicados, derivar variables binarias del campo `notas`,
preparar la variable objetivo y dejar definido el protocolo de evaluación
(partición estratificada / validación cruzada) que utilizarán los notebooks
de modelado.

## Entrada y salida
- **Entrada:** `data/propiedades.csv` — 60 propiedades en venta de 6 colonias,
  recolectadas manualmente de Inmuebles24 (Fase 1) y analizadas en
  `01_eda_inicial.ipynb` (Fase 2).
- **Salida:** conjunto de modelado con las características construidas y la
  partición/CV definida, para `03_modelo_baseline.ipynb` y
  `04_modelo_comparativo.ipynb`.

## Nota sobre el tamaño de muestra
Con n=60, este trabajo es una prueba de concepto metodológica, no un modelo de
valuación desplegable. Las decisiones priorizan un flujo sin fuga de información
y estimaciones honestas por validación cruzada por encima de la métrica puntual.

## Contenido (provisional)
1. Configuración
2. Carga y validación del contrato de datos
3. Construcción del conjunto de modelado (eliminación de duplicados)
4. Ingeniería de características desde `notas`
5. Preparación de la variable objetivo
6. Protocolo de evaluación (partición estratificada / CV)
7. Prueba estadística de preventa
8. Exportación del conjunto de modelado

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)  # ver todas las columnas al inspeccionar

# El notebook vive en notebooks/; los datos en data/ (un nivel arriba).
# Resolvemos la raíz para que la ruta no dependa del directorio de ejecución.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "propiedades.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("¿Existe el CSV?:", DATA_PATH.exists())

Raíz del proyecto: /Users/edson/Documents/GitHub/dataset-inmuebles-merida
¿Existe el CSV?: True


## 2. Carga y validación del contrato de datos
Antes de transformar algo, se confirma que el archivo real coincide con el
contrato documentado (forma, columnas, tipos, faltantes) y se inspecciona la
estructura categórica que condicionará la estratificación y el test de preventa.
Celda de solo lectura: no modifica los datos.

In [5]:
# Validación del contrato de datos (solo lectura, no modifica nada)
COLUMNAS_ESPERADAS = [
    "fecha_registro", "url", "url_archivo", "operacion", "tipo_inmueble",
    "colonia", "precio", "m2_construccion", "m2_terreno", "recamaras",
    "banos", "estacionamientos", "antigüedad", "es_preventa", "notas",
]
COLUMNAS_NUMERICAS = [
    "precio", "m2_construccion", "m2_terreno", "recamaras",
    "banos", "estacionamientos", "antigüedad",
]

df = pd.read_csv(DATA_PATH)

print("Forma:", df.shape)  # esperado: (60, 15)
print("Columnas en orden esperado:", list(df.columns) == COLUMNAS_ESPERADAS)

print("\nTipos inferidos:")
print(df.dtypes)

print("\nFaltantes por columna:")
print(df.isna().sum())

print("\nCeros en columnas numéricas (revisar si alguno debería ser NaN):")
print((df[COLUMNAS_NUMERICAS] == 0).sum())

print("\nPor colonia:")
print(df["colonia"].value_counts())
print("\nPor tipo_inmueble:")
print(df["tipo_inmueble"].value_counts())
print("\nPor es_preventa:")
print(df["es_preventa"].value_counts())

mask_dup = df["notas"].str.contains("posible duplicado", case=False, na=False)
print(f"\nFilas marcadas como posible duplicado: {int(mask_dup.sum())}")
print(df.loc[mask_dup, ["colonia", "precio", "notas"]])

Forma: (60, 15)
Columnas en orden esperado: False

Tipos inferidos:
fecha_registro          str
url                     str
url_archivo         float64
operación               str
tipo_inmueble           str
colonia                 str
precio                int64
m2_construccion       int64
m2_terreno            int64
recamaras             int64
banos                 int64
estacionamientos    float64
antigüedad          float64
es_preventa             str
notas                   str
dtype: object

Faltantes por columna:
fecha_registro       0
url                  0
url_archivo         60
operación            0
tipo_inmueble        0
colonia              0
precio               0
m2_construccion      0
m2_terreno           0
recamaras            0
banos                0
estacionamientos     1
antigüedad          39
es_preventa          0
notas                1
dtype: int64

Ceros en columnas numéricas (revisar si alguno debería ser NaN):
precio              0
m2_construccion     0
m2_ter